In [1]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

In [4]:
# filepath to the .excel file
# Load the dataset from the cloned GitHub repository

filepath = 'https://github.com/Knchna/Credit_Card_Default_Prediction/blob/main/default%20of%20credit%20card%20clients.xls'
df = pd.read_excel(filepath, header=1)
df.head(20)

ValueError: Excel file format cannot be determined, you must specify an engine manually.

In [ ]:
df.head()

In [ ]:
df = df.drop(columns=["ID"])

In [ ]:
X = df.drop(columns=["default payment next month"])

y = df["default payment next month"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=45,
    stratify=y
)

In [ ]:
categorical_features = [
    "SEX",
    "EDUCATION",
    "MARRIAGE",
    "PAY_0",
    "PAY_2",
    "PAY_3",
    "PAY_4",
    "PAY_5",
    "PAY_6"
]

numerical_features = [
    "LIMIT_BAL",
    "AGE",
    "BILL_AMT1",
    "BILL_AMT2",
    "BILL_AMT3",
    "BILL_AMT4",
    "BILL_AMT5",
    "BILL_AMT6",
    "PAY_AMT1",
    "PAY_AMT2",
    "PAY_AMT3",
    "PAY_AMT4",
    "PAY_AMT5",
    "PAY_AMT6"
]

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_features
        ),
        (
            "cat",
            "passthrough",
            categorical_features
        )
    ]
)

In [ ]:
logistic_model = LogisticRegression(
    C=1,
    penalty="l1",
    solver="liblinear",
    class_weight="balanced",
    max_iter=1000,
    random_state=45
)

In [ ]:
model_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", logistic_model)
    ]
)

In [ ]:
model_pipeline.fit(X_train, y_train)

In [ ]:
y_pred = model_pipeline.predict(X_test)

In [ ]:
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

In [ ]:
probabilities = model_pipeline.predict_proba(X_test)

default_probability = probabilities[:, 1]

In [ ]:
model_pipeline.predict_proba(X_test.iloc[[0]])

In [ ]:
joblib.dump(
    model_pipeline,
    "credit_default_pipeline.pkl"
)

In [ ]:
loaded_pipeline = joblib.load(
    "credit_default_pipeline.pkl"
)

In [ ]:
prediction = loaded_pipeline.predict(X_test.iloc[[0]])

probability = loaded_pipeline.predict_proba(
    X_test.iloc[[0]]
)

In [ ]:
grid_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

In [ ]:
param_grid = {
    "model__C": [0.01, 0.1, 1, 10],
    "model__penalty": ["l1", "l2"],
    "model__solver": ["liblinear"],
    "model__class_weight": [None, "balanced"]
}